In [7]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests
import warnings

warnings.filterwarnings('ignore')

def create_heatmap(input_csv, output_pdf, stat_csv):
    sns.set_theme(style="ticks", context="paper")
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
    plt.rcParams['pdf.fonttype'] = 42

    df = pd.read_csv(input_csv)

    donor_cols = [c for c in df.columns if c.startswith('HS-')]
    df[donor_cols] = df[donor_cols].apply(pd.to_numeric, errors='coerce')
    df = df.dropna(subset=donor_cols)

    control_row = df[df['KULFFI'] == 'Control']
    if control_row.empty:
        raise ValueError("Control row not found in the dataset.")

    control_vals = control_row[donor_cols].values[0].astype(float)
    df_materials = df[df['KULFFI'] != 'Control'].copy()

    df_materials['NUtrision classfication'] = df_materials['NUtrision classfication'].replace(
        {'Monosaccharides & Polyols': 'Monosaccharides/Polyols'}
    )

    mat_vals = df_materials[donor_cols].values.astype(float)

    p_values = []
    for i in range(mat_vals.shape[0]):
        try:
            stat, p = ttest_rel(mat_vals[i], control_vals)
        except Exception:
            p = 1.0
        if np.isnan(p): p = 1.0
        p_values.append(p)

    df_materials['P_value'] = p_values
    reject, q_values, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')
    df_materials['Q_value'] = q_values

    stats_df = df_materials[['KULFFI', 'P_value', 'Q_value']]
    stats_df.to_csv(stat_csv, index=False)

    epsilon = 0.01
    log2fc_vals = np.log2((mat_vals + epsilon) / (control_vals + epsilon))

    df_log2fc = pd.DataFrame(log2fc_vals, columns=donor_cols, index=df_materials['KULFFI'])
    df_log2fc['Category'] = df_materials['NUtrision classfication'].values

    category_order = [
        'Polysaccharides', 'Oligosaccharides', 'Disaccharides', 'Monosaccharides/Polyols',
        'Amino acids', 'Organic acids', 'Minerals',
        'Water-soulble vitamins', 'Fat-soulble vitamins', 'Vitamins-like substances',
        'Polyphenols', 'Lipids', 'Fresh/Processed foods',
        'Pharmaceuticals/Supplemants', 'Other food components'
    ]

    color_map = {
        'Polysaccharides': '#8B0000',
        'Oligosaccharides': '#DC143C',
        'Disaccharides': '#FF6347',
        'Monosaccharides/Polyols': '#FFC0CB',
        'Amino acids': '#7CFC00',
        'Organic acids': '#3CB371',
        'Minerals': '#808080',
        'Water-soulble vitamins': '#00FFFF',
        'Fat-soulble vitamins': '#1E90FF',
        'Vitamins-like substances': '#00BFFF',
        'Polyphenols': '#A0522D',
        'Lipids': '#FFD700',
        'Fresh/Processed foods': '#D2B48C',
        'Pharmaceuticals/Supplemants': '#EAFF00',
        'Other food components': '#000000'
    }

    material_colors = df_log2fc['Category'].map(color_map).fillna('white')
    heatmap_data = df_log2fc.drop(columns=['Category']).T
    material_colors = material_colors.reindex(heatmap_data.columns)
    material_colors.name = ""
    heatmap_data.columns.name = ""

    g = sns.clustermap(heatmap_data,
                       col_colors=material_colors,
                       cmap='bwr',
                       center=0,
                       vmin=-1.0, vmax=1.0,
                       square=True,
                       figsize=(42, 14),
                       xticklabels=False,
                       yticklabels=False,
                       dendrogram_ratio=(0.1, 0.15),
                       cbar_pos=None,
                       tree_kws={'linewidths': 1.0})

    if g.ax_col_colors:
        g.ax_col_colors.set_xlabel('')
        g.ax_col_colors.set_ylabel('')
        g.ax_col_colors.set_xticks([])
        g.ax_col_colors.set_yticks([])
        g.ax_col_colors.tick_params(axis='both', which='both', length=0)
        for spine in g.ax_col_colors.spines.values():
            spine.set_visible(False)

    for ax in g.fig.axes:
        if ax.get_ylabel() in ["Category", "NUtrision classfication", ""]:
            ax.set_ylabel("")
            ax.set_yticks([])
            ax.set_xticks([])
            ax.tick_params(axis='both', which='both', length=0)
            for spine in ax.spines.values():
                spine.set_visible(False)

    for spine in g.ax_heatmap.spines.values():
        spine.set_visible(False)

    g.ax_heatmap.set_xlabel('KULFFI (177 ingredients)', fontsize=38, fontweight='bold', labelpad=30)
    g.ax_heatmap.set_ylabel('', fontsize=38, fontweight='bold', labelpad=30)

    legend_elements = [Patch(facecolor=color_map.get(cat, 'white'), edgecolor='black', label=cat)
                       for cat in category_order if cat in color_map]

    legend = g.ax_heatmap.legend(handles=legend_elements, title='Ingredient Category',
                        bbox_to_anchor=(1.02, 1.15),
                        loc='upper left',
                        borderaxespad=0.,
                        frameon=False,
                        fontsize=24,
                        title_fontsize=28)
    legend.get_title().set_fontweight('bold')

    cbar_ax = g.fig.add_axes([1.02, 0.05, 0.03, 0.25])
    cb = plt.colorbar(g.ax_heatmap.collections[0], cax=cbar_ax)
    cbar_ax.set_ylabel('Log2 Fold Change', fontsize=32, fontweight='bold', labelpad=30)
    cbar_ax.tick_params(labelsize=24)

    reordered_cols = g.dendrogram_col.reordered_ind
    reordered_materials = heatmap_data.columns[reordered_cols]

    targets = ['Erythritol', 'Xylitol', 'Inulin', 'Mannose', 'Resistant maltodextrin', 'Resistant starch']
    target_pos = {}
    for target in targets:
        if target in reordered_materials:
            idx = list(reordered_materials).index(target)
            target_pos[target] = idx

    y_min, y_max = g.ax_heatmap.get_ylim()
    bottom_y = max(y_min, y_max)

    y_offsets_orig = {
        'Resistant starch': 7.0,
        'Resistant maltodextrin': 10.0,
        'Inulin': 10.0,
        'Xylitol': 11.0,
        'Erythritol': 14.0,
        'Mannose': 14.0
    }

    shift_val = 2.5
    shifts_orig = {}
    sorted_targets = sorted([(idx, t) for t, idx in target_pos.items()])

    for i, (idx, t) in enumerate(sorted_targets):
        shifts_orig[t] = 0
        if i > 0:
            prev_idx, prev_t = sorted_targets[i-1]
            if idx - prev_idx < 3:
                 shifts_orig[t] = shift_val
                 shifts_orig[prev_t] = -shift_val

    final_shifts = shifts_orig.copy()
    final_y_offsets = y_offsets_orig.copy()

    def fix_crossing(t1, t2, t_pos, f_shifts, f_offsets, off1, off2):
        if t1 in t_pos and t2 in t_pos:
            idx1 = t_pos[t1]
            idx2 = t_pos[t2]
            if abs(idx1 - idx2) < 5:
                if idx1 < idx2:
                    f_shifts[t1] = -3.5
                    f_shifts[t2] = 3.5
                else:
                    f_shifts[t1] = 3.5
                    f_shifts[t2] = -3.5

            f_offsets[t1] = off1
            f_offsets[t2] = off2

    # Prevent crossing for RS and RMD
    fix_crossing('Resistant starch', 'Resistant maltodextrin', target_pos, final_shifts, final_y_offsets, 7.5, 9.5)

    # Align Erythritol and Xylitol starting positions perfectly with Mannose (Y offset = 14.0)
    fix_crossing('Erythritol', 'Xylitol', target_pos, final_shifts, final_y_offsets, 14.0, 14.0)

    y_turn_offset = 2.0
    char_height_unit = 2.5
    gap = 4.0

    for target, x_idx in target_pos.items():
        x_col = x_idx + 0.5
        if target == 'Resistant maltodextrin':
            label_text = 'RMD'
        elif target == 'Resistant starch':
            label_text = 'RS'
        else:
            label_text = target

        shift_x = final_shifts.get(target, 0)
        x_text = x_col + shift_x

        this_offset = final_y_offsets.get(target, 12.0)
        y_text_base = bottom_y + this_offset

        text_len_units = len(label_text) * char_height_unit
        y_text_top_visual = y_text_base - text_len_units
        y_line_end = y_text_top_visual - gap

        y_turn = bottom_y + y_turn_offset
        if y_line_end < y_turn + 0.5:
             y_line_end = y_turn + 0.5

        if shift_x == 0:
            xs = [x_col, x_col]
            ys = [bottom_y, y_line_end]
        else:
            xs = [x_col, x_col, x_text, x_text]
            ys = [bottom_y, y_turn, y_turn, y_line_end]

        g.ax_heatmap.plot(xs, ys, color='black', linewidth=1.5, clip_on=False)

        g.ax_heatmap.text(
            x_text,
            y_text_base,
            label_text,
            rotation=90,
            va='bottom',
            ha='center',
            fontsize=32,
            fontweight='bold',
            color='black'
        )

    plt.savefig(output_pdf, dpi=600, bbox_inches='tight')
    plt.close()

# Generate Heatmaps
create_heatmap('Acetate(mM).csv', 'Figure_4a_Acetate.pdf', 'Acetate_Statistics.csv')
create_heatmap('Butyrate(mM).csv', 'Figure_4a_Butyrate.pdf', 'Butyrate_Statistics.csv')